# Práctica: Comparativa de clustering (k-means vs HC vs DBSCAN)

Objetivo: comparar **K-Means**, **Clustering Jerárquico Aglomerativo (HC)** y **DBSCAN** en 3 conjuntos de datos sintéticos de `sklearn` diseñados para que, en general:

- Dataset A (**blobs esféricos**) → suele “ganar” **k-means**
- Dataset B (**two moons / no convexo**) → suele “ganar” **DBSCAN**
- Dataset C (**anisotrópico + densidades distintas**) → suele “ganar” **HC** (con linkage adecuado)

**Métrica de comparación principal:** *Adjusted Rand Index (ARI)* usando las etiquetas verdaderas (como los datos son sintéticos).  
> Nota: en problemas reales no supervisados no tendrás `y_true`, pero aquí lo usamos para comparar métodos.


## Tarea: buscar la mejor configuración (HC, K-Means y DBSCAN)

En los tres métodos vas a implementar la misma idea:

1. **Definir un conjunto de parámetros a probar** (`param_grid`).

```python
param_grid = {
    "n_clusters": [1,2,3,4,5,6]     ,
    "linkage": ...,
    ...
}
```

2. **Recorrer todas las combinaciones** con `ParameterGrid(param_grid)`.

```python
for params in ParameterGrid(param_grid):
   ...
```

3. Para cada combinación:
   - Crear el modelo con `**params`
   - Obtener las etiquetas con `fit_predict(X_a_scaled)`
   - Calcular el **ARI** comparando con `y_a`

```python
    model = **********(**params)              # ******** AgglomerativeClustering; KMeans o DBSCAN
    labels = model.fit_predict(X_a_scaled)
    ari = adjusted_rand_score(y_a, labels)
```

4. Guardar la mejor configuración:
   - `best_ari`
   - `best_params`
5. Re-entrenar el modelo con `best_params` y guardar:
   - `cluster_labels_*` (según el método)

### Pistas generales

- Usa siempre el dataset escalado: **`X_a_scaled`**
- La métrica de evaluación será: **`adjusted_rand_score(y_a, labels)`**
- Mantén una estructura tipo:
  - `best = (-1, None)`
  - `if ari > best[0]: best = (ari, params)`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import ParameterGrid

### Funciones auxiliares comunes

In [ ]:
def plot_clusters_2d(X, labels, title, title_inf=None):
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, s=12)
    plt.title(title)
    # Texto inferior (centrado abajo)
    # Mostrar subtítulo solo si se proporciona
    if title_inf is not None:
        plt.figtext(0.5, 0.01, title_inf, ha="center", fontsize=10)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.tight_layout()
    plt.show()

### Conjunto 1. Datos esfericos y bien separados

In [ ]:
# Generación
X_a, y_a = make_blobs(
    n_samples=900,
    centers=4,
    cluster_std=0.60,
    random_state=0
)

# Escalado
X_a_scaled = StandardScaler().fit_transform(X_a)

plot_clusters_2d(X_a_scaled, y_a, "Dataset A (verdad): blobs esféricos (escalado)")

## Resolución con Hierarchical Clustering

En esta sección realizamos una búsqueda en cuadrícula (grid search) sobre distintos parámetros del algoritmo de Agglomerative Clustering para encontrar la configuración que mejor se ajusta a la estructura real del dataset.

El objetivo será seleccionar la combinación de parámetros que maximiza el Adjusted Rand Index (ARI), comparando los clusters obtenidos con las etiquetas verdaderas del conjunto sintético.

### Parámetros explorados

- **n_clusters**: número de grupos considerados (2 a 6).
- **linkage**:
  - ward
  - complete
  - average
  - single
- **metric**:
  - euclidean
  - manhattan
  - cosine

Nota: El método `ward`  solo es compatible con distancia euclídea, por lo que se excluyen automáticamente combinaciones inválidas.

Finalmente guarda en:

-  best_ari: el mejor valor de ARI encontrado.
-  best_params: la configuración de parámetros correspondiente.
-  cluster_labels_hc: las etiquetas del mejor modelo.

In [ ]:
# Configurar clustering jerárquico


In [ ]:
plot_clusters_2d(X_a_scaled, cluster_labels_hc, f"Dataset A: HC {best_params}", f"HC ARI: {best_ari}")

## Resolución con K-Means

En esta sección realizamos una búsqueda en cuadrícula (grid search) sobre distintos parámetros del algoritmo K-Means para encontrar la configuración que mejor se ajusta a la estructura real del dataset.

El objetivo será seleccionar la combinación de parámetros que maximiza el Adjusted Rand Index (ARI), comparando los clusters obtenidos con las etiquetas verdaderas del conjunto sintético.

### Parámetros explorados

- **n_clusters**: número de grupos considerados (2 a 6).
- **init**:
  - k-means++
  - random
- **n_init**: número de inicializaciones independientes del algoritmo.
- **max_iter**: número máximo de iteraciones permitidas.
- **tol**: tolerancia para el criterio de convergencia.
- **random_state**: semilla para garantizar reproducibilidad.

Nota: A diferencia del clustering jerárquico, K-Means optimiza explícitamente la suma de cuadrados intra-cluster (SSE) y puede converger a distintos mínimos locales dependiendo de la inicialización.

Finalmente guarda en:

- **best_ari**: el mejor valor de ARI encontrado.
- **best_params**: la configuración de parámetros correspondiente.
- **cluster_labels_kmeans**: las etiquetas del mejor modelo.

In [ ]:
# Configurar KMeans


In [ ]:
plot_clusters_2d(X_a_scaled, cluster_labels_kmeans, f"Dataset A: K-means {best_params}", f"K-means ARI: {best_ari}")

## Resolución con DBSCAN

En esta sección realizamos una búsqueda en cuadrícula (grid search) sobre distintos parámetros del algoritmo DBSCAN para identificar la configuración que mejor captura la estructura de densidad presente en el dataset.

El objetivo será seleccionar la combinación de parámetros que maximiza el Adjusted Rand Index (ARI), comparando los clusters obtenidos con las etiquetas verdaderas del conjunto sintético.

### Parámetros explorados

- **eps**: radio máximo de vecindad para considerar puntos como vecinos.
- **min_samples**: número mínimo de puntos necesarios para que una región se considere densa.
- **metric**:
  - euclidean
  - manhattan
  - cosine

Nota: A diferencia de K-Means y Hierarchical Clustering, DBSCAN no requiere fijar previamente el número de clusters.  
El número final de grupos depende de los parámetros seleccionados y puede incluir puntos etiquetados como ruido (`-1`).

Durante la búsqueda se descartan configuraciones degeneradas, como aquellas que generan un único cluster o clasifican todos los puntos como ruido.

Finalmente guarda en:

- **best_ari**: el mejor valor de ARI encontrado.
- **best_params**: la configuración de parámetros correspondiente.
- **cluster_labels_dbscan**: las etiquetas del mejor modelo.

In [ ]:
# Configurar DBSCAN



In [ ]:
plot_clusters_2d(X_a_scaled, cluster_labels_DBSCAN, f"Dataset A: DBSCAN {best_params}", f"DBSCAN ARI: {best_ari}")

## Dataset B: Two Moons (no convexo) con ruido (suele ganar DBSCAN)

**Idea:** dos grupos no convexos (medias lunas) + ruido.  
- k-means tiende a “cortar” cada luna.
- HC depende del *linkage* (single puede encadenar; ward/complete tienden a fallar).
- DBSCAN suele funcionar muy bien si `eps` está bien elegido.

In [ ]:
X_b, y_b = make_moons(n_samples=900, noise=0.06, random_state=0)

X_b_scaled = StandardScaler().fit_transform(X_b)

plot_clusters_2d(X_b_scaled, y_b, "Dataset B (verdad): two moons (escalado)")

### Evalua los métodos de aprendizaje no supervisado 

## Dataset C: Anisotrópico + densidades diferentes (suele favorecer HC)

**Idea:** construir un dataset donde:
- Los clusters son elípticos (anisotrópicos).
- Además hay densidades distintas (un cluster muy denso y otro más disperso).

Efectos:
- **k-means** puede partir mal regiones elípticas o imponer fronteras “esféricas”.
- **DBSCAN** sufre si hay densidades distintas: un único `eps` no sirve para todos.
- **HC** con linkage *average* o *complete* puede ser más estable aquí (según parámetros).

In [ ]:
# Generamos 3 blobs con desviaciones distintas (densidades diferentes)
X_c1, y_c1 = make_blobs(n_samples=350, centers=[(-2, 2)], cluster_std=0.35, random_state=1)  # denso
X_c2, y_c2 = make_blobs(n_samples=350, centers=[(2, 0)],  cluster_std=0.75, random_state=2)  # disperso
X_c3, y_c3 = make_blobs(n_samples=350, centers=[(6, 3)],  cluster_std=0.45, random_state=3)  # medio

A1 = np.array([[2.0, -2.5],
    [0.5,  0.8]])
A2 = np.array([[0.6, -0.8],
              [0.4,  3.9]])

# Aplicamos transformación lineal
X_c2t = X_c2 @ A1
X_c1t = X_c1 @ A2

X_c = np.vstack([X_c1t, X_c2t, X_c3])
y_c = np.array([0]*len(X_c1) + [1]*len(X_c2) + [2]*len(X_c3))


# Escalado
X_c_scaled = StandardScaler().fit_transform(X_c)

plot_clusters_2d(X_c_scaled, y_c, "Dataset C (verdad): anisotrópico + densidades distintas (escalado)")